# 13. Agents

A ReAct agent that uses a web-search tool and a weather tool.

> Setup: `pip install -r requirements.txt` and copy `.env.example` to `.env` first.

In [ ]:
from dotenv import load_dotenv

load_dotenv()  # reads OPENAI_API_KEY (and others) from the .env file in the repo root

> Needs `OPENAI_API_KEY` and a free [Weatherstack](https://weatherstack.com/) key as `WEATHERSTACK_API_KEY` in `.env`.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
import os
import requests

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

In [ ]:
@tool
def get_weather_data(city: str) -> dict:
    """Fetch the current weather data for a given city."""
    api_key = os.environ["WEATHERSTACK_API_KEY"]
    response = requests.get(
        "https://api.weatherstack.com/current",
        params={"access_key": api_key, "query": city},
        timeout=10,
    )
    response.raise_for_status()
    return response.json()

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [ ]:
from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub

In [ ]:
# Step 2: Pull the ReAct prompt from LangChain Hub
prompt = hub.pull("hwchase17/react")  # pulls the standard ReAct agent prompt

In [ ]:
# Step 3: Create the ReAct agent manually with the pulled prompt
agent = create_react_agent(
    llm=llm,
    tools=[search_tool, get_weather_data],
    prompt=prompt
)

In [ ]:
# Step 4: Wrap it with AgentExecutor
agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool, get_weather_data],
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=8,
)

In [ ]:
# Step 5: Invoke
response = agent_executor.invoke({"input": "Find the capital of Madhya Pradesh, then find its current weather condition"})
print(response)

In [ ]:
response['output']